In [1]:
!pip install transformers==4.41.0 -q
print('Instalado.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 66.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.5 MB/s eta 0:00:00:00:01
Instalado.


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.naive_bayes import ComplementNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('GPU:', torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [5]:
train = pd.read_csv('/kaggle/input/datasets/beleniniestatejera/trabajo/train_clean.csv')
test = pd.read_csv('/kaggle/input/datasets/beleniniestatejera/prueba/test_nolabel.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print()
print('Distribución label:')
print(train['label'].value_counts())

Train shape: (8947, 10)
Test shape: (3836, 7)

Distribución label:
label
1    5794
0    3153
Name: count, dtype: int64


In [6]:
# Preparar texto combinando speaker, partido y statement
def preparar_texto(df):
    textos = []
    for _, row in df.iterrows():
        texto = f"Speaker: {row['speaker']}. Party: {row['party_affiliation']}. Statement: {row['statement']}"
        textos.append(texto)
    return textos

train_textos = preparar_texto(train)
test_textos = preparar_texto(test)

# Split train/validación
train_textos_tr, train_textos_val, y_tr, y_val = train_test_split(
    train_textos, train['label'].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=train['label']
)

labels_val_all = np.array(y_val)

print(f'Train: {len(train_textos_tr)} filas')
print(f'Val:   {len(train_textos_val)} filas')
print(f'Test:  {len(test_textos)} filas')
print()
print(f'Ejemplo: {train_textos[0]}')

Train: 7604 filas
Val:   1343 filas
Test:  3836 filas

Ejemplo: Speaker: donald-trump. Party: republican. Statement: China is in the South China Sea and a military fortress the likes of which perhaps the world has not seen.


In [7]:
tokenizer_large = RobertaTokenizer.from_pretrained('roberta-large')

class FakeNewsDataset(Dataset):
    def __init__(self, textos, labels=None, tokenizer=None, max_length=128):
        self.textos = textos
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.textos[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze()
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = FakeNewsDataset(train_textos_tr, y_tr, tokenizer_large)
val_dataset = FakeNewsDataset(train_textos_val, y_val, tokenizer_large)
test_dataset = FakeNewsDataset(test_textos, tokenizer=tokenizer_large)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f'Train: {len(train_dataset)} ejemplos, {len(train_loader)} batches')
print(f'Val:   {len(val_dataset)} ejemplos')
print(f'Test:  {len(test_dataset)} ejemplos')

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

Train: 7604 ejemplos, 951 batches
Val:   1343 ejemplos
Test:  3836 ejemplos


In [8]:
model_large = RobertaForSequenceClassification.from_pretrained(
    'roberta-large',
    num_labels=2
)
model_large = model_large.to(device)

optimizer_large = AdamW(model_large.parameters(), lr=1e-5, weight_decay=0.01)

epochs_large = 3  # sabemos que epoch 3 es el óptimo
total_steps_large = len(train_loader) * epochs_large
scheduler_large = get_linear_schedule_with_warmup(
    optimizer_large,
    num_warmup_steps=int(0.1 * total_steps_large),
    num_training_steps=total_steps_large
)

print(f'Modelo cargado en: {device}')
print(f'Parámetros totales: {sum(p.numel() for p in model_large.parameters()):,}')
print(f'Epochs: {epochs_large}')
print(f'Total steps: {total_steps_large}')

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modelo cargado en: cuda
Parámetros totales: 355,361,794
Epochs: 3
Total steps: 2853


In [10]:
def evaluar(model, loader):
    model.eval()
    preds, labels_all = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=-1)
            
            preds.extend(predictions.cpu().tolist())
            labels_all.extend(labels.cpu().tolist())
    
    f1 = f1_score(labels_all, preds, average='macro')
    return f1, preds, labels_all

best_f1_large = 0

for epoch in range(epochs_large):
    model_large.train()
    total_loss = 0
    
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer_large.zero_grad()
        outputs = model_large(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model_large.parameters(), 1.0)
        optimizer_large.step()
        scheduler_large.step()
        total_loss += loss.item()
        
        if batch_idx % 150 == 0:
            print(f'Epoch {epoch+1}/{epochs_large} | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}')
    
    avg_loss = total_loss / len(train_loader)
    val_f1, _, _ = evaluar(model_large, val_loader)
    
    print(f'\nEpoch {epoch+1} completado | Avg Loss: {avg_loss:.4f} | Val F1-Macro: {val_f1:.4f}')
    
    if val_f1 > best_f1_large:
        best_f1_large = val_f1
        torch.save(model_large.state_dict(), 'best_model_large.pt')
        print(f'Mejor modelo guardado! F1: {best_f1_large:.4f}')
    print()

Epoch 1/3 | Batch 0/951 | Loss: 0.7467
Epoch 1/3 | Batch 150/951 | Loss: 0.5071
Epoch 1/3 | Batch 300/951 | Loss: 0.8166
Epoch 1/3 | Batch 450/951 | Loss: 0.4809
Epoch 1/3 | Batch 600/951 | Loss: 0.3284
Epoch 1/3 | Batch 750/951 | Loss: 0.8165
Epoch 1/3 | Batch 900/951 | Loss: 0.6681

Epoch 1 completado | Avg Loss: 0.6479 | Val F1-Macro: 0.6267
Mejor modelo guardado! F1: 0.6267

Epoch 2/3 | Batch 0/951 | Loss: 0.6120
Epoch 2/3 | Batch 150/951 | Loss: 0.9013
Epoch 2/3 | Batch 300/951 | Loss: 0.5391
Epoch 2/3 | Batch 450/951 | Loss: 0.6694
Epoch 2/3 | Batch 600/951 | Loss: 0.5535
Epoch 2/3 | Batch 750/951 | Loss: 0.8117
Epoch 2/3 | Batch 900/951 | Loss: 0.8360

Epoch 2 completado | Avg Loss: 0.5860 | Val F1-Macro: 0.6114

Epoch 3/3 | Batch 0/951 | Loss: 0.4394
Epoch 3/3 | Batch 150/951 | Loss: 0.5729
Epoch 3/3 | Batch 300/951 | Loss: 0.6956
Epoch 3/3 | Batch 450/951 | Loss: 0.4436
Epoch 3/3 | Batch 600/951 | Loss: 0.3118
Epoch 3/3 | Batch 750/951 | Loss: 0.1801
Epoch 3/3 | Batch 900/951 

In [11]:
# Cargar mejor modelo
model_large.load_state_dict(torch.load('best_model_large.pt'))
model_large.eval()

# Probabilidades en test y validación
proba_large_test = []
proba_large_val = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model_large(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=-1)
        proba_large_test.extend(probs[:, 1].cpu().tolist())

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model_large(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=-1)
        proba_large_val.extend(probs[:, 1].cpu().tolist())

proba_large_test = np.array(proba_large_test)
proba_large_val = np.array(proba_large_val)

print(f'Probabilidades test: {proba_large_test.shape}')
print(f'Probabilidades val:  {proba_large_val.shape}')

Probabilidades test: (3836,)
Probabilidades val:  (1343,)


In [13]:
# Reconstruir NB mejorado
MIN_COUNT = 5
vc = train['speaker'].value_counts()
speakers_frecuentes = vc[vc >= MIN_COUNT].index
test_speaker_grouped = test['speaker'].where(test['speaker'].isin(speakers_frecuentes), other='other')

vc_party = train['party_affiliation'].value_counts()
minoritarias = vc_party[vc_party < 50].index
test_party_grouped = test['party_affiliation'].where(~test['party_affiliation'].isin(minoritarias), other='other')
test_subject_primary = test['subject'].str.split(',').str[0].str.strip()

cat_cols = ['speaker_grouped', 'party_grouped', 'subject_primary']
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_cat = ohe.fit_transform(train[cat_cols])

test_cat_df = pd.DataFrame({
    'speaker_grouped': test_speaker_grouped,
    'party_grouped': test_party_grouped,
    'subject_primary': test_subject_primary
})
X_test_cat = ohe.transform(test_cat_df)

vect_nb = CountVectorizer(ngram_range=(1, 2), binary=True, min_df=7)
X_train_nb = hstack([vect_nb.fit_transform(train['statement']), X_train_cat])
X_test_nb = hstack([vect_nb.transform(test['statement']), X_test_cat])

nb_model = ComplementNB(alpha=3.5, norm=False)
nb_model.fit(X_train_nb, train['label'])

proba_nb_test = nb_model.predict_proba(X_test_nb)[:, 1]
proba_nb_val = nb_model.predict_proba(X_train_nb[-len(labels_val_all):])[: ,1]

print('NB listo.')
print()

# Probar ensemble RoBERTa-large + NB
print('Ensemble RoBERTa-large + NB:')
print()
for w_large in [0.80, 0.85, 0.90, 0.95]:
    w_nb = round(1 - w_large, 2)
    proba_ens_val = w_large * proba_large_val + w_nb * proba_nb_val
    
    for threshold in [0.45, 0.50, 0.55]:
        y_pred = (proba_ens_val >= threshold).astype(int)
        f1 = f1_score(labels_val_all, y_pred, average='macro')
        print(f'Large={w_large:.2f} NB={w_nb:.2f} threshold={threshold}: F1-Val={f1:.4f}')
    print()

NB listo.

Ensemble RoBERTa-large + NB:

Large=0.80 NB=0.20 threshold=0.45: F1-Val=0.6326
Large=0.80 NB=0.20 threshold=0.5: F1-Val=0.6454
Large=0.80 NB=0.20 threshold=0.55: F1-Val=0.6384

Large=0.85 NB=0.15 threshold=0.45: F1-Val=0.6396
Large=0.85 NB=0.15 threshold=0.5: F1-Val=0.6468
Large=0.85 NB=0.15 threshold=0.55: F1-Val=0.6466

Large=0.90 NB=0.10 threshold=0.45: F1-Val=0.6422
Large=0.90 NB=0.10 threshold=0.5: F1-Val=0.6469
Large=0.90 NB=0.10 threshold=0.55: F1-Val=0.6457

Large=0.95 NB=0.05 threshold=0.45: F1-Val=0.6457
Large=0.95 NB=0.05 threshold=0.5: F1-Val=0.6486
Large=0.95 NB=0.05 threshold=0.55: F1-Val=0.6466



In [14]:
print('Afinar pesos ensemble:')
print()
for w_large in [0.92, 0.93, 0.94, 0.95, 0.96, 0.97]:
    w_nb = round(1 - w_large, 2)
    proba_ens_val = w_large * proba_large_val + w_nb * proba_nb_val
    
    for threshold in [0.48, 0.50, 0.52]:
        y_pred = (proba_ens_val >= threshold).astype(int)
        f1 = f1_score(labels_val_all, y_pred, average='macro')
        print(f'Large={w_large:.2f} NB={w_nb:.2f} threshold={threshold}: F1-Val={f1:.4f}')
    print()

Afinar pesos ensemble:

Large=0.92 NB=0.08 threshold=0.48: F1-Val=0.6454
Large=0.92 NB=0.08 threshold=0.5: F1-Val=0.6440
Large=0.92 NB=0.08 threshold=0.52: F1-Val=0.6463

Large=0.93 NB=0.07 threshold=0.48: F1-Val=0.6454
Large=0.93 NB=0.07 threshold=0.5: F1-Val=0.6440
Large=0.93 NB=0.07 threshold=0.52: F1-Val=0.6495

Large=0.94 NB=0.06 threshold=0.48: F1-Val=0.6474
Large=0.94 NB=0.06 threshold=0.5: F1-Val=0.6466
Large=0.94 NB=0.06 threshold=0.52: F1-Val=0.6508

Large=0.95 NB=0.05 threshold=0.48: F1-Val=0.6478
Large=0.95 NB=0.05 threshold=0.5: F1-Val=0.6486
Large=0.95 NB=0.05 threshold=0.52: F1-Val=0.6508

Large=0.96 NB=0.04 threshold=0.48: F1-Val=0.6442
Large=0.96 NB=0.04 threshold=0.5: F1-Val=0.6479
Large=0.96 NB=0.04 threshold=0.52: F1-Val=0.6485

Large=0.97 NB=0.03 threshold=0.48: F1-Val=0.6415
Large=0.97 NB=0.03 threshold=0.5: F1-Val=0.6499
Large=0.97 NB=0.03 threshold=0.52: F1-Val=0.6495



In [15]:
# Submission RoBERTa-large solo
y_pred_large_solo = (proba_large_test >= 0.50).astype(int)
submission_solo = pd.DataFrame({'id': test['id'], 'label': y_pred_large_solo})
submission_solo.to_csv('submission_roberta_large.csv', index=False)
print('Submission RoBERTa-large solo guardada.')
print('Distribución:', dict(submission_solo['label'].value_counts()))

print()

# Submission ensemble RoBERTa-large + NB
proba_ensemble = 0.95 * proba_large_test + 0.05 * proba_nb_test
y_pred_ensemble = (proba_ensemble >= 0.52).astype(int)
submission_ens = pd.DataFrame({'id': test['id'], 'label': y_pred_ensemble})
submission_ens.to_csv('submission_roberta_large_ensemble.csv', index=False)
print('Submission ensemble guardada.')
print('Distribución:', dict(submission_ens['label'].value_counts()))

print()
print('Resumen:')
print(f'  RoBERTa-large solo:     F1-Val=0.6482 | threshold=0.50')
print(f'  Ensemble Large+NB:      F1-Val=0.6508 | threshold=0.52 | Large=0.95 NB=0.05')

Submission RoBERTa-large solo guardada.
Distribución: {1: np.int64(2458), 0: np.int64(1378)}

Submission ensemble guardada.
Distribución: {1: np.int64(2396), 0: np.int64(1440)}

Resumen:
  RoBERTa-large solo:     F1-Val=0.6482 | threshold=0.50
  Ensemble Large+NB:      F1-Val=0.6508 | threshold=0.52 | Large=0.95 NB=0.05
